# Data Wrangling with Spark

These first three cells import libraries, instantiate a SparkSession, and then read in the data set

In [ ]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import desc
from pyspark.sql.functions import asc
from pyspark.sql.functions import sum as Fsum

In [ ]:
%matplotlib inline

In [ ]:
spark = SparkSession \
    .builder \
    .appName("Wrangling Data") \
    .getOrCreate()

#### For this exercise we will download a much larger dataset that will be used in the following exercises

In [ ]:

# download from google storage
import requests

url = 'https://storage.googleapis.com/files-bb-bot/music_log.json'
r = requests.get(url)
with open('./data/music_log.json', 'wb') as f:
    f.write(r.content)

In [ ]:
path = "data/music_log.json"
user_log = spark.read.json(path)

In [ ]:
## You'll see that this has much more entries
print(user_log.count())

# Data Exploration 

The next cells explore the data set.

In [ ]:
user_log.take(5)

In [ ]:
user_log.printSchema()

In [ ]:
user_log.describe("sessionId").show()

In [ ]:
user_log.count()

Now, in order to know the distinct pages in the dataset, we can use the following approach. 

By selecting the `page` column, removing duplicate entries with `dropDuplicates()`, and sorting the results alphabetically using `sort("page")`, we get a clean list of all unique page names. Finally, `show()` displays the output.

If you're looking for alternatives, you could use:
```python
user_log.select("page").distinct().orderBy("page").show()

In [ ]:
user_log.select("page").dropDuplicates().sort("page").show()

In [ ]:
user_log.select("page").distinct().orderBy("page").show()

### Query Data

Now, if we want to filter the dataset to see all records where the `userId` is "100001", we can use this query. It selects the columns `userId`, `firstname`, `page`, and `song`, applies a filter with `.where()`, and retrieves the data using `.collect()`.

This approach is useful when you want to:
1. Focus on specific columns of interest.
2. Examine all data associated with a particular `userId`.
3. Fetch the filtered rows as a list of Python objects (since `.collect()` pulls the data into memory).

If you don't want to load all results into memory, you could use `.show()` instead of `.collect()` to preview the data directly in the console:
```python
user_log.select(["userId", "firstname", "page", "song"]).where(user_log.userId == "100001").show()

In [ ]:
user_log.select(["userId", "firstname", "page", "song"]).where(user_log.userId == "100001").collect()

# Calculating Statistics by Hour

Here, we define a user-defined function (UDF) named `get_hour`. This UDF takes a timestamp in milliseconds as input and converts it into an hour value.
The conversion is achieved by dividing the timestamp by 1000 to convert it into seconds, and then using the `datetime` module to extract the hour from the resulting timestamp.

UDFs like this are commonly used in Spark when you need to apply custom transformations that are not natively supported.

In this case, the goal is likely to extract the hour component from a timestamp column in the dataset to perform time-based analysis or aggregations.

In [ ]:
from pyspark.sql.functions import udf
get_hour = udf(lambda x: datetime.datetime.fromtimestamp(x / 1000.0).hour)

Here, we are creating a new column named `hour` in the `user_log` DataFrame. This column is derived by applying the previously defined `get_hour` UDF to the `ts` column (which likely contains timestamps in milliseconds). The result is the hour value extracted from the timestamp.

This step is useful for adding a time-based feature to the DataFrame, enabling analysis or grouping by hour to uncover patterns or trends based on the time of day.

In [ ]:
user_log = user_log.withColumn("hour", get_hour(user_log.ts))

In [ ]:
user_log.show(n=1)

In this example, we add a new column named `logs_hour` to the `user_log` DataFrame. 
Instead of using a UDF, we leverage Spark's built-in functions:

- `from_unixtime`: Converts the timestamp (in milliseconds) into a human-readable format by dividing it by 1000 to change it to seconds.
- `hour`: Extracts the hour component from the resulting timestamp.
- `col("ts")`: Refers to the `ts` column in the DataFrame.

In [ ]:
from pyspark.sql.functions import hour, from_unixtime, col
logs_hours = user_log.withColumn("logs_hour", hour(from_unixtime(col("ts") / 1000)))
logs_hours.show(n=1)

Now ife we want to calculate the number of songs played during each hour.

This is achieved by filtering the `user_log` DataFrame to include only rows where the `page` column has the value `"NextSong"`, which likely represents a song play event.

The steps are as follows:

1. **Filter**: Retain only rows where `page == "NextSong"`.
2. **GroupBy**: Group the filtered DataFrame by the `hour` column.
3. **Count**: Count the number of rows (songs played) in each hour group.
4. **OrderBy**: Sort the resulting groups by the `hour` column, cast as a float for proper numerical ordering.

This operation results in a DataFrame showing how many songs were played in each hour of the day, sorted by the hour in ascending order.

In [ ]:
songs_in_hour = (user_log
                 .filter(user_log.page == "NextSong")
                 .groupby(user_log.hour)
                 .count()
                 .orderBy(user_log.hour.cast("float")))

In [ ]:
songs_in_hour.show()

Convert the `songs_in_hour` Spark DataFrame into a Pandas DataFrame using the `toPandas()` method, which allows us to work with the data locally in Pandas.

After converting, we explicitly cast the `hour` column in the Pandas DataFrame to a numeric type using `pd.to_numeric`. This ensures that the `hour` values are treated as numbers, which is useful if they were initially strings or objects in the Pandas DataFrame.

This is typically done to enable numerical operations, such as plotting or further processing, on the `hour` column.

In [ ]:
songs_in_hour_pd = songs_in_hour.toPandas()
songs_in_hour_pd.hour = pd.to_numeric(songs_in_hour_pd.hour)

In [ ]:
songs_in_hour_pd[songs_in_hour_pd['hour'] == 2]

### Plotting the data

Create a scatter plot visualizing the number of songs played during each hour of the day.

In [ ]:
plt.scatter(songs_in_hour_pd["hour"], songs_in_hour_pd["count"])
plt.xlim(-1, 24);
plt.ylim(0, 1.2 * max(songs_in_hour_pd["count"]))
plt.xlabel("Hour")
plt.ylabel("Songs played");

# Drop Rows with Missing Values

As you'll see, it turns out there are no missing values in the userID or session columns. But there are userID values that are empty strings.

In [ ]:
## order by sessionId ascending
user_log.select("userId").dropDuplicates().sort("userId").show(5)

### Drop the empty ones

We create a new DataFrame `user_log_valid` by removing rows with missing values in the `user_log` DataFrame.

The `dropna()` method is used to handle missing (null) values.

1. **`how="any"`**:
   - This means that if **any** of the specified columns (`"userId"` or `"sessionId"`) contain a null value, the row will be dropped.
   - Other possible options for `how`:
     - `"all"`: Drops rows only if **all** specified columns are null.

2. **`subset=["userId", "sessionId"]`**:
   - Specifies the columns to check for null values. Only rows with nulls in these columns are considered for removal.

In [ ]:
user_log_valid = user_log.dropna(how = "any", subset = ["userId", "sessionId"])

Ensure that all rows with an empty or whitespace-only `userId` are removed from the `user_log_valid` DataFrame.

In [ ]:
from pyspark.sql.functions import trim
user_log_valid = user_log_valid.filter(trim(user_log_valid["userId"]) != "")

In [ ]:
user_log_valid.count()

In [ ]:
user_log.count()

In [ ]:
user_log_valid.select("userId").dropDuplicates().sort("userId").show(5)

# Tracking User Account Downgrades and Behavior Across Lifetime”

Now, if we look at our dataset, we know that at any given timestamp, a user is either a paid or free user based on the `level` column. However, this information alone doesn't tell us whether the user has previously upgraded or downgraded their account. Without this context, we cannot track the transitions users make between account levels, such as identifying when a user actively downgraded their account from paid to free.

To track these transitions, we can flag specific log entries, such as those where a user submits a downgrade request (`page = "Submit Downgrade"`). By using a window function and a cumulative sum, we can identify and segment a user's activity into distinct phases:
- **Pre-downgrade phase**: All activity before the first downgrade event.
- **Post-downgrade phase(s)**: Activity after one or more downgrade events.

This segmentation allows us to analyze user behavior before and after downgrading, providing insights into patterns that might predict downgrades or indicate how user engagement changes after transitioning to a free account.


Find when users downgrade their accounts and then flag those log entries. 

Then use a window function and cumulative sum to distinguish each user's data as either pre or post downgrade events.

In [ ]:
user_log_valid.filter("page = 'Submit Downgrade'").show()

In [ ]:
user_log.select(["userId", "firstname", "page", "level", "song"]).where(user_log.userId == "100025").collect()

Define a User-Defined Function (UDF) called `flag_downgrade_event`, which is used to create a binary flag for identifying downgrade events in the dataset.

In [ ]:
flag_downgrade_event = udf(lambda x: 1 if x == "Submit Downgrade" else 0, IntegerType())

Add a new column called `downgraded` to the `user_log_valid` DataFrame by applying the `flag_downgrade_event` UDF to the `page` column.

In [ ]:
user_log_valid = user_log_valid.withColumn("downgraded", flag_downgrade_event("page"))

In [ ]:
user_log_valid.show(5)

Import the `Window` module from PySpark to define window specifications for performing advanced operations, such as cumulative sums, ranking, or aggregations, within partitions of data.

Window functions allow calculations across a subset of rows related to the current row, grouped by specific criteria (e.g., `userId`) and ordered by columns (e.g., timestamps).

In [ ]:
from pyspark.sql import Window

Define a window specification `windowval` to calculate cumulative metrics for each user (`userId`) based on their activity logs. 

**Details:**

1. **`partitionBy("userId")`**:
   - Groups rows by `userId`, ensuring calculations are performed independently for each user.

2. **`orderBy(desc("ts"))`**:
   - Orders the rows for each user in descending order of the `ts` (timestamp) column, meaning the most recent activity comes first.

3. **`rangeBetween(Window.unboundedPreceding, 0)`**:
   - Specifies the range of rows considered for each calculation:
     - `Window.unboundedPreceding`: Includes all rows from the start of the partition.
     - `0`: Includes the current row.
   - Calculation includes all rows up to and including the current row, creating a cumulative effect.

In [ ]:
windowval = Window.partitionBy("userId").orderBy(desc("ts")).rangeBetween(Window.unboundedPreceding, 0)

Create a new column `phase` by applying a cumulative sum of the `downgraded` column over the defined window. 
This assigns a value of `0` for all rows before the first downgrade event and increments by `1` for each downgrade event, segmenting user activity into distinct phases.

`Fsum` is a PySpark function that computes a cumulative sum over a specified window. 
In this case, it adds up the values in the `downgraded` column, which contains `1` for downgrade events and `0` otherwise. This allows us to track how many downgrades have occurred up to each row, creating a running total within the defined window.

In [ ]:
user_log_valid = user_log_valid.withColumn("phase", Fsum("downgraded").over(windowval))

Now, if we take, for example, the user with ID `100025`, we will see that their actions are split into those that happened before and after the downgrade event. These actions are also categorized by their account level and the specific pages they visited, giving us a clear timeline of their behavior.

In [ ]:
user_log_valid.select(["userId", "firstname", "ts", "page", "level", "phase"]).where(user_log.userId == "100025").sort("ts", ascending=True).show(800)

Now we also see that there are some users with a `phase` value equal to `2`, which means they have downgraded their account twice. This allows us to track their behavior across multiple transitions, examining their actions before and after each downgrade event to understand their engagement and account activity patterns over time.

In [ ]:
user_log_valid.select(["userId", "firstname", "ts", "page", "level", "phase"]).where(user_log.userId == "100004").sort("ts", ascending=True).show(800)